In [1]:
#!/usr/bin/env python3
"""
Convert a Jupyter notebook (.ipynb) into a LaTeX snippet.

- Every code cell   -> wrapped in \\begin{mintedbox}{python} ... \\end{mintedbox}
- Every markdown cell -> turned into a \\section{...} headline
  (leading '#' markdown heading markers are stripped; the remaining
  text is flattened onto one line and used as the section title)

Usage:
    python notebook_to_latex.py input.ipynb output.tex
"""

import json
import sys


def cell_source_to_text(cell) -> str:
    """nbformat stores 'source' as either a string or a list of lines."""
    source = cell.get("source", "")
    if isinstance(source, list):
        return "".join(source)
    return source


def markdown_to_title(text: str) -> str:
    """Turn a markdown cell's text into a single-line section title."""
    # Strip leading markdown heading symbols (#, ##, ...) if present
    text = text.strip()
    if text.startswith("#"):
        text = text.lstrip("#").strip()
    # Collapse everything onto a single line
    text = " ".join(line.strip() for line in text.splitlines() if line.strip())
    return text


def notebook_to_latex(notebook_path: str, output_path: str) -> None:
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    blocks = []
    for cell in nb.get("cells", []):
        cell_type = cell.get("cell_type")
        text = cell_source_to_text(cell).strip()

        if not text:
            continue  # skip empty cells

        if cell_type == "code":
            blocks.append(
                f"\\begin{{mintedbox}}{{python}}\n{text}\n\\end{{mintedbox}}"
            )
        elif cell_type == "markdown":
            title = markdown_to_title(text)
            if title:
                blocks.append(f"\\section{{{title}}}")
        # any other cell types (raw, etc.) are ignored for now

    latex_output = "\n\n".join(blocks) + "\n"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(latex_output)

    print(f"Wrote {len(blocks)} block(s) to {output_path}")


if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python notebook_to_latex.py <input.ipynb> <output.tex>")
        sys.exit(1)

    notebook_to_latex(sys.argv[1], sys.argv[2])


Usage: python notebook_to_latex.py <input.ipynb> <output.tex>


SystemExit: 1

/Users/gernotstuhler/miniforge3_arm64/envs/scFates/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
notebook_to_latex("Tabula Sapiens.ipynb", "Sapiens.tex")

Wrote 116 block(s) to Sapiens.tex
